## Running circuits with Qiskit primitives

### The Sampler primitive

A sampler allows you to recover the output probabilities of computational basis states at the end of your computation. It is useful in contexts where we are interested in the amplitudes of specific states, such as in Grover's algorithm or when solving combinatorial optimization problems.

Qiskit contains many implentations of ```Sampler``` classes, each appropriate for different contexts. For noiseless quantum simulation, Qiskit primitives offer a simple and easy to use implementation. 

In [ ]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

The sampler class has a method, ```run()```, which takes as input a (or a list of) quantum circuit(s) from which we wish to recover output probabilities. Be careful! When using a sampler, it is necessary to add a measurement instruction to the circuit(s) being sampled. This can be done through the ```.measure()``` method by specifying which qubits to measure (and the output classical register), or simply with the ```measure_all()``` method if measuring every qubit.

```Sampler().run()``` returns a ```PrimitiveJob``` object. Along with the result of the computation, this object contains metadata about its execution, that we won't bother with for now. 

Here, we are intrested in the quasi-probability distribution of basis states after our computation. With ```PrimitiveJob().result()```, we fetch a ```SamplerPubResult``` object. The results for each circuit passed to the sampler can be obtained by indexing ```SamplerPubResult```. Each resultant object has a data attribute, from which we can obtain the values in the classical register via using the register name and the ```get_counts()``` method. 

In [ ]:
from qiskit import QuantumCircuit
import numpy as np

# Prepare circuit #1
qc1 = QuantumCircuit(2)
qc1.h(0)
qc1.cx(0, 1)
qc1.measure_all()
print("Circuit #1:")
print(qc1.draw())

# Prepare circuit #2
qc2 = QuantumCircuit(4, 3)
for i in range(4):
    qc2.h(i)
qc2.ry(np.pi / 3, 2)
qc2.measure([1, 2, 3], [0, 1, 2])


print("")
print("")
print("")
print("Circuit #2:")
print(qc2.draw())

# Execute computation using the Sampler
job = sampler.run([qc1, qc2])
# Fetch job result
result = job.result()



counts=[]
# Classical register is called 'meas' in the first circuit and 'c' in the second circuit, so we need to access them separately
counts.append(result[0].data.meas.get_counts())
counts.append(result[1].data.c.get_counts())

total_shots = []
total_shots.append(sum(counts[0].values()))
total_shots.append(sum(counts[1].values()))

# Calculate probabilities for each circuit, based on the counts and total shots
probabilities = []
probabilities.append({k: v / total_shots[0] for k, v in counts[0].items()})
probabilities.append({k: v / total_shots[1] for k, v in counts[1].items()})

print("")
print("")
print("")
# Measurement probabilities of first circuit
print("Measurement probabilities of first circuit:")
print(probabilities[0])

# Measurement probabilities of second circuit
print("Measurement probabilities of second circuit:")
print(probabilities[1])

As you may know, quantum computers cannot reveal exact probabilities: The current best way around this is to estimate by sampling the circuit many times. Doing so induces *shot noise* in the estimate, which scales as the inverse square root of the number of repetitions.

To emulate a real quantum computation, it is possible to pass to the sampler a specific number of shots. When doing so, we ask Qiskit to add shot noise to the simulation results.

In [ ]:
# Initialize a new Sampler
sampler_shots = StatevectorSampler()

job_shots = sampler_shots.run([qc2], shots=100)

result = job_shots.result()
counts = result[0].data.c.get_counts()
probabilities = {k: v / sum(counts.values()) for k, v in counts.items()}

print("Measurement probabilities of second circuit with shots=100:")
print(probabilities)

When working with samplers, the typical way to visualize computation results is via a histogram. Qiskit offers a built-in method to construct histograms out of the counts.

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts)

### The Estimator primitive

Other than the counts and their associated probabilities, another quantity we may be interested in at the end of our quantum computation is the expectation value of some observables. While the sampler is in general sufficient for this purpose, measuring expectation values of complicated operators can take many circuits, with many changes of bases. Qiskit can do this work behind the scenes for us if we make use of the ```Estimator``` primitive. 

Much like the sampler, the estimator takes for input a (or a list of) quantum circuit(s), however we also need to specify an observable , or a list of operators, for which to compute expectation values. For simple Pauli string observables, it is sufficient to pass a string. Be mindful of the little endian convention: the observable ```'ZX'``` corresponds to $X$ measured on qubit $0$, and $Z$ on qubit $1$.

Unlike the Sampler, when using an Estimator you should NOT add measurement instructions to your circuit. The estimator takes care of adding the measurement operation to the circuit. Any measurement you put in the circuit will induce a wavefunction collapse and change the measurement outcome.

To pass the circuit to an estimator, we create primitive unit blocs (PUBs). Each PUB is a tuple of (circuit, list[observables]).

A list of PUBs is then passed to the estimator.

The output is similar to that of the sampler, but with the expectation values obtained with ```evs``` instead of the classical register name.

In [ ]:
from qiskit.primitives import StatevectorEstimator

# Instantiate an estimator
estim = StatevectorEstimator()

# prepare quantum circuit
qc = QuantumCircuit(2)
qc.h(0)

# Evaluate exp. val. of X on qubit 0, Z on qubit 1
pub = (qc, ["ZX"]) # Primitive Unit Bloc (PUB): a tuple of (circuit, observables)
job = estim.run([pub])
# Fetch exp. val.
print(job.result()[0].data.evs)

When working with more complicated observables, multiple avenues are possible. If you know its decomposition in terms of Pauli strings, as is often the case, best is to use a ```SparsePauliOp``` object which can be passed directly to the estimator. An observable $O = \sum_{i} \alpha_i \mathcal{P}_i$ can be instantiated by passing a list of strings $[\mathcal{P}_0, \mathcal{P}_1, ...]$ and a list of coefficients $[\alpha_0, \alpha_1,...]$.

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# create observable obs = ZX + 2YY + 3II
obs = SparsePauliOp(["ZX", "YY", "II"], [1, 2, 3])

# measure exp. val. on circuit
pub = (qc, obs)
job = estim.run([pub])

print(job.result()[0].data.evs)

```SparsePauliOp``` also implements a basic algebra, thus more complicated observables can be constructed by composing simpler ones using this algebra.

This basic algebra notably includes addition and subtraction with operators, multiplication with scalars and operators, and the tensor product of operators.

In [ ]:
obs2 = obs @ obs  # operator product
print("May contain repetitions:", obs2)
# we can observe repeated strings in the sparse structure
# The simplify method eliminates such repetition, and it can also remove elements with coefficients close to 0
obs2 = (obs2 + 1e-16 * SparsePauliOp("IX")).simplify(atol=1e-14, rtol=1e-14)
print("After simplification:",obs2)

pub2 = (qc, obs2)
job = estim.run([pub2])

print(job.result()[0].data.evs)

### Running Parameterized Circuits

When executing a parameterized circuit, it is necessary to specify a value for the parameters upon execution. While the parameters can be assigned with ```assign_parameters()``` as seen previously, this can also be done directly by passing a list of values as an argument to the ```.run()``` method. This method does not accept dictionaries as input, you must pass it a list of values. Be mindful of the ordering!

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

# 1. Prepare parameterized quantum circuit
theta = ParameterVector("θ", 3)
phi = Parameter("φ")

param_qc = QuantumCircuit(3)
for i in range(3):
    param_qc.ry(theta[i], i)
param_qc.cry(phi, 1, 2)
param_qc.measure_all()

# Inspect parameter ordering
# Output order will be: (ParameterVectorElement(θ[0]), ParameterVectorElement(θ[1]), ParameterVectorElement(θ[2]), Parameter(φ))
print("Circuit parameters:", param_qc.parameters)

# 2. Format parameter values as a flat array matching the exact order above
# θ = [1, 2, 3] and φ = 4 -> [1, 2, 3, 4]
parameter_values = [1, 2, 3, 4]

# 3. Create the proper Sampler PUB tuple
# Format: (circuit, parameter_values)
pub = (param_qc, parameter_values)

# 4. Run the Sampler
sampler = StatevectorSampler()
job = sampler.run([pub])
result = job.result()

# 5. Extract counts and plot
# Classical register is named 'meas' by measure_all()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

# Plot the histogram directly using the counts dictionary
plot_histogram(counts)

### Exercise

The Heisenberg model Hamiltonian is a well known model of magnetic systems. It is given by:

\begin{equation}
\mathcal{H} = -\frac{1}{2} \sum_{j = 0}^{N-1} J_x X_j X_{j+1} + J_y Y_j Y_{j+1} + J_z Z_j Z_{j+1}
\end{equation}

where $X_j = I^{\otimes j-1} \otimes X \otimes I^{\otimes N-j-1}$ is the Pauli string containing an $X$ at position $j$ and $I$ everywhere else (same goes for $Y_j$ and $Z_j$). Here, we are considering a one-dimensional spin chain with periodic boundary conditions, which means the indices are taken modulo $N$.

The following circuit prepares a quantum state $\ket{\psi}$ of a 5-site spin chain, where every qubit corresponds to the spin at one site.

  ![circuit1](./images/efficientSU2.png)

Assume the geometry of the spin chain corresponds to the geometry of the circuit, i.e. neighbouring qubits are interacting, with qubits 0 and 4 interacting due to periodicity. Your task is to build the circuit shown above, and to evaluate the energy $\bra{\psi} \mathcal{H} \ket{\psi}$ for the provided parameter values as well as the following interaction constants:

\begin{equation}
J_x = 0.73, \hspace{1cm} J_y = 0.99, \hspace{1cm} J_z = 0.07
\end{equation}

Make use of qiskit's circuit ```TwoLocal```to build your quantum circuit.

In [ ]:
rng = np.random.default_rng(seed=42)
thetas = 2 * np.pi * rng.random(30)
from qiskit.circuit.library import TwoLocal

print(thetas)
###Your code here###

## Running circuits with Qiskit IBM Runtime

So far, all our calculations have been done on noiseless simulators. To run quantum algorithms on real hardware (or noisy simulators), it is necessary to go through another library, ```qiskit_ibm_runtime```.

### Backend 

When performing real quantum computations, one must keep in mind the hardware on which the computation will be performed. Indeed, many factors such as qubit connectivity, chip geometry and  the set of available native gates, which may all differ between individual quantum processors, will affect the quality of results obtained for a specific calculation. In Qiskit, you may specify which processor you want to use (as long as its access has been granted to you) by instantiating a ```Backend``` object.

For the purpose of this tutorial, we will be using a ```FakeBackend```, which emulates the connectivity and noise of a specific IBM processor. To use real hardware, you must go through a specific provider.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeQuebec

# Instantiate the fake backend, which imitates the Eagle processor in the IBM Lagos computer
backend = FakeQuebec()

# from here, we may obtain some information about the backend, such as:
# number of qubits
print(backend.num_qubits)
# the properties, such as T1, T2 in microseconds, of a specific qubit
print(backend.qubit_properties(5))
# the readout error of measurement on a specific qubit
print(backend.target["measure"][(5,)])

### Transpilation

The image below shows the connectivity of the Eagle r3 processor.


<img src="./images/quebec_connectivity.png" alt="connectivity" width="600"/>

(The colors of the qubits and their connections communicate information about their readout and two-qubit gate error rate, respectively). 

Unfortunately, superconducting quantum processors can only execute two-qubit gates between adjacent qubits. For most relevant medium-scale quantum algorithms, where there might be large numbers of two qubit gates between many pairs of qubits, the quantum circuit must be reinterpreted to be compatible with the geometry of the chip. Furthermore, not every gate can be directly implemented on hardware level. Some gates in your circuit may need to be be "translated" into the chip's native gateset. 


This entire process is called "transpilation". Before sending a job to a backend, the circuit MUST be transpiled specifically for it! 

"Transpilation" is similar to the conversion of classical programming languages like C into the machine code that a specific CPU can understand and execute.

The simplest way to transpile a quantum circuit is through a ```PassManager``` object. 

Due to the limitations of current quantum computers, it is desirable to execute our algorithms in the least amount of operations possible. During the transpilation process, the pass manager can look for cancellations or simplifications in the quantum circuit, as well as look for "tricks" on the hardware level. However, there is a nontrivial tradeoff between the optimization of the circuit and the time necessary for transpilation. Therefore, when instantiating a pass manager, you should specify the target backend, as well as the desired optimization level between $0$ and $3$, $0$ being no optimization at all and $3$ being the maximal amount.

Depending on optimization level, the pass manager may automatically select which qubits on the chip are best adapted for your specific circuit, based on required connectivity and qubit quality.


In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Generate a preset pass manager with no optimization
pm_0 = generate_preset_pass_manager(backend=backend, optimization_level=0)

# make a test circuit. Let's create a circuit which prepares a GHZ state
qc = QuantumCircuit(5)
qc.h(0)
for i in range(4):
    qc.cx(i, i + 1)
qc.draw("mpl")

In [ ]:
# perform transpilation
transpiled_qc_0 = pm_0.run(qc)
# Note that transpiled_qc is now a 127-qubit quantum circuit, however 122 of those qubits are idle
transpiled_qc_0.draw("mpl", idle_wires=False)

What are we looking at? Notice:
* $q_n \to m$ indicates that the nth qubit in the circuit was mapped to qubit number m on the chip. Here, the pass manager uses a trivial mapping $n = m$
* CNOT is not native to the fake IBM Québec. Two-qubit gates are implemented using the ECR operation.
* Single qubit gates are a combination of $\sqrt{X}, X$ and $R_z$ gates

Now, let's compare with a heavily optimized transpilation

In [ ]:
pm_3 = generate_preset_pass_manager(backend=backend, optimization_level=3)
transpiled_qc_3 = pm_3.run(qc)
transpiled_qc_3.draw("mpl", idle_wires=False)
# As you can see, the circuit is much shallower, and the pass manager chose the chip qubits more carefully.

Now that our circuit has been transpiled, we are ready to send it to the backend. In this case, we must use the ```Sampler``` and ```Estimator``` from ```qiskit_ibm_runtime```, which imply slight differences in syntax.

### SamplerV2
A crucial difference between the ```qiskit``` and ```qiskit_ibm_runtime``` primitives is that you may pass to the runtime primitives execution options. This includes the number of shots (or observable target precision for the Estimator), as well as error correction and mitigation techniques which you may want to use. These options are passed through a special object, either a ```SamplerOptions``` or an ```EstimatorOptions```.

Options for error correction and mitigation include:
* Readout mitigation
* Dynamical decoupling
* Zero Noise Extrapolation (ZNE)
* Pauli twirling

You are invited to consult the [Qiskit documentation](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques) for more details.

For SamplerV2, PUBs are of the form ```(QuantumCircuit, [parameter values], shots (Optional))```.

For a specific job using a SamplerV2, ```job.result()[i].data``` contains a ```DataBin``` object of the measurement data for the $i^\text{th}$ PUB. To obtain the counts dictionary, use ```job.result()[i].data.cr.get_counts()```, where "cr" is the name of the classical register you are storing your measurement information in. When using ```measure_all()```, the default name of the classical register is ```meas```.

In [ ]:
from qiskit_ibm_runtime import SamplerOptions, SamplerV2

# Instantiate Sampler, with 1000 shots
sampler_opt = SamplerOptions()
sampler_opt.default_shots = 1000

samplerV2 = SamplerV2(mode=backend, options=sampler_opt)

# Run our transpiled GHZ circuit. First, we add measurements to the circuit
qc.measure_all()
# then, transpile
transpiled_qc_3_sampler = pm_3.run(qc)
# execute job
job = samplerV2.run([transpiled_qc_3_sampler, transpiled_qc_3_sampler])
# plot measurement data
plot_histogram(job.result()[0].data.meas.get_counts())
# The presence of noise becomes obvious from the histogram

### EstimatorV2


For EstimatorV2, PUBs take the form ```(QuantumCircuit, [observables], [parameter values])```. Furthermore, the observable(s) must be transpiled with the circuit layout, so that the correct observables are measured on the correct qubits. The layout of your circuit can be accessed in the ```circuit.layout``` attribute, and the observable may be transpiled by using the ```apply_layout()``` method.

For estimator jobs, the ```DataBin``` contains an attibute ```evs``` which contains the expectation values of all the observables in the PUB.

In [ ]:
from qiskit_ibm_runtime import EstimatorOptions, EstimatorV2
from qiskit.quantum_info import SparsePauliOp

# Instantiate an estimator with a target precision of 0.001
estim_opt = EstimatorOptions()
estim_opt.default_precision = 0.001

estimV2 = EstimatorV2(mode=backend, options=estim_opt)

# measure ZZZZZ on GHZ circuit. We expect to obtain 0
observable = SparsePauliOp("ZZZZZ")
isa_obs = observable.apply_layout(transpiled_qc_3.layout)
isa_obs

# evaluate expval
job = estimV2.run([(transpiled_qc_3, isa_obs)])
print(job.result()[0].data.evs)

# Again, due to noise, the expval is not exactly 0.

### Exercise

You are given a hidden black box circuit $\mathcal{U}$, which prepares an unknown state $\mathcal{U}\ket{0} = \ket{\psi}$. Using the ```EfficientSU2``` parameterized ansatz from ```qiskit.circuit.library```, your objective is to build a circuit $\mathcal{U'_{\vec{\theta}}}$ such that $\mathcal{U'_{\vec{\theta}}}\ket{0} = \ket{\psi'}$ approximates $\ket{\psi}$ as closely as possible. Your ansatz should have exactly 3 layers, which corresponds to 40 parameters. To optimize your prameters $\vec{\theta}$, you should define a cost function which computes the fidelity of both states: $F(\psi, \psi') = |\langle \psi | \psi' \rangle|^2$ on the provided backend, and you should try to minimize the quantity $1 - F$ using ```scipy.optimize.minimize```. You are encouraged to explore different solvers and optimization options.

Although it probably will not be of any help, to enhance your immmersion you are invited not to "peek" inside the black box in any way, for example by drawing or decomposing $\mathcal{U}$.

Your main method should return a list of parameters, as well as the fidelity you obtain with these parameters.

In [ ]:
from scipy.optimize import minimize
from utils import black_box
from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit_aer import QasmSimulator


def cost(parameter_values, U: QuantumCircuit, backend, options):

    ### Your code here ###

    return


def main():
    backend = QasmSimulator()
    options = SamplerOptions()
    options.simulator.seed_simulator = int(hash("summer school")) % 10**5
    U = black_box()

    ### Your code here###

    return


main()